# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAlmasri7/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [16]:
import os
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train",
    token=os.getenv("HF_TOKEN")
)

print(dataset)

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [17]:
from itertools import islice
import pandas as pd

sample = pd.DataFrame(islice(dataset, 5))
sample

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


### Unit of Analysis

For the **Ranking Signal Analysis** lane, I use the **fact_content_daily_performance** table.

Each row represents **one content page for one client on one report date**. The table contains daily search performance and engagement signals that can be used to analyze the relationship between ranking signals and page performance.

### Time Window

**March 2026 (month = '2026-03')**

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# Fields

## Features

The following five features will be used because they are available when an SEO specialist makes a decision.

| Feature | Available at the decision moment because... |
|----------|---------------------------------------------|
| gsc_impressions | Search visibility has already been observed. |
| gsc_clicks | Historical clicks have already been recorded. |
| gsc_sum_position | Historical ranking information is already available. |
| scroll_events | User scrolling behaviour has already been measured. |
| sessions_ai | AI-generated traffic has already been observed. |

## Label / Proxy

The warehouse does not contain a predefined target label such as "declining page."

Therefore, I define a **proxy target**.

A page will be considered **declining** if its search impressions decrease between two consecutive observation windows. This proxy represents pages whose visibility is decreasing and may require SEO review.

The proxy is derived from observed historical performance rather than manually assigned labels.

## Context

These columns identify each observation but are not predictive features.

- report_date
- client_hash_id
- content_hash_id

## Excluded

| Field | Reason |
|--------|--------|
| client_hash_id | Identifier only; not useful for prediction. |
| content_hash_id | Identifier only; not useful for prediction. |
| client_has_gsc | Describes data availability rather than ranking behaviour. |
| client_has_ga4 | Describes data availability rather than ranking behaviour. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()

token = os.getenv("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

In [23]:
rel = "hf://datasets/FlyRank/internship-warehouse"

In [24]:
march = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

In [25]:
df = march.df()

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [26]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT (
            CAST(report_date AS VARCHAR)
            || client_hash_id
            || content_hash_id
        )
    ) AS unique_rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,total_rows,unique_rows
0,9841378,9841378


The total number of rows should equal the number of unique (report_date, client, content) combinations, confirming that one row represents one content page for one client on one report date.

In [27]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


This query verifies the size of my selected slice and confirms that all observations belong to March 2026.

In [28]:
con.sql(f"""
SELECT
    gsc_data_available,
    COUNT(*) AS rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY gsc_data_available
ORDER BY gsc_data_available DESC
""").df()

,gsc_data_available,rows
0,True,3611061
1,False,6230317


This query verifies how many observations have Google Search Console data available and confirms that my analysis uses valid observations.

In [34]:
feature_frame = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    scroll_events,
    sessions_ai
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
LIMIT 20
""").df()

feature_frame

,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0
5,5,0,27,0,0
6,0,0,0,0,0
7,0,0,0,0,0
8,39,0,221,0,1
9,179,0,923,0,1


In [35]:
feature_frame.describe()

,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai
count,20.000000,20.000000,20.000000,20.0,20.000000
mean,189.500000,0.150000,1200.000000,0.0,0.200000
std,729.239438,0.366348,4491.803541,0.0,0.410391
min,0.000000,0.000000,0.000000,0.0,0.000000
25%,0.000000,0.000000,0.000000,0.0,0.000000
50%,2.500000,0.000000,13.500000,0.0,0.000000
75%,39.250000,0.000000,329.750000,0.0,0.000000
max,3282.000000,1.000000,20243.000000,0.0,1.000000


## Why these features?

### gsc_impressions
Knowable at the decision moment because search visibility has already been measured.

### gsc_clicks
Knowable because historical search clicks already exist before making any optimisation decision.

### gsc_sum_position
Knowable because historical ranking information has already been collected.

### scroll_events
Knowable because user scrolling behaviour has already been recorded.

### sessions_ai
Knowable because AI-driven traffic has already occurred before the optimisation decision.

In [36]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS gsc_impressions,
    COUNT(gsc_clicks) AS gsc_clicks,
    COUNT(gsc_sum_position) AS gsc_sum_position,
    COUNT(scroll_events) AS scroll_events,
    COUNT(sessions_ai) AS sessions_ai
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,total_rows,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai
0,9841378,9841378,9841378,9841378,6822637,6822637


**Observation**

The selected Google Search Console features are available for nearly all observations, while engagement-related features have lower availability.

Choosing features with higher completeness makes the data contract more reliable for later modelling.

In [46]:
experiment = feature_frame.copy()

experiment["proxy_label"] = (
    experiment["gsc_clicks"] > 0
).astype(int)

In [47]:
honest_features = experiment[
    [
        "gsc_impressions",
        "gsc_sum_position",
        "scroll_events",
        "sessions_ai"
    ]
]

honest_features.head()

,gsc_impressions,gsc_sum_position,scroll_events,sessions_ai
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0
4,0,0,0,0


The honest feature set deliberately excludes gsc_clicks because it is used to define the proxy label.

In [48]:
experiment["leaked_feature"] = experiment["proxy_label"]

leaked_features = experiment[
    [
        "gsc_impressions",
        "gsc_sum_position",
        "scroll_events",
        "sessions_ai",
        "leaked_feature"
    ]
]

leaked_features.head()

,gsc_impressions,gsc_sum_position,scroll_events,sessions_ai,leaked_feature
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


In [49]:
experiment.drop(columns="leaked_feature", inplace=True)

## Feature Leakage

The proxy label was created from `gsc_clicks`.

The honest feature set excludes `gsc_clicks`, ensuring that the future model would rely only on historical search signals.

To demonstrate feature leakage, I intentionally added a new feature copied directly from the proxy label.

If this feature were used during model training, it would leak the correct answer into the inputs, producing unrealistically optimistic evaluation results.

The leaked feature was then removed so that future modelling uses only information that would genuinely be available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This warehouse contains observational search performance data.

It can identify relationships between search signals and content performance, but it cannot prove causal relationships or explain Google's ranking algorithm.

External factors such as algorithm updates, competitor actions, seasonality, and editorial decisions are not represented in the data.

In addition, some Google Analytics fields are unavailable for a large proportion of observations, so this notebook primarily relies on Google Search Console signals.

The findings should therefore be interpreted as observed, measured, directional, and decision-support rather than causal proof.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.